# The `%%Mexplodemodel` Jupyter Magic

The `%%Mexplodemodel` (aliased as `%%mdmodel`) cell magic lets you write **model specifications directly in notebook cells** using Markdown-style text mixed with ModelFlow DSL equations. Under the hood it parses the cell, feeds everything to `Mexplode`, and pushes the resulting object into the notebook namespace — ready to use.

This notebook explains how it works and walks through all the options.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import modeljupytermagic          # registers the magics
from modelconstruct import Mexplode

## 1. Basic syntax

The first line after `%%Mexplodemodel` carries the **model name** and optional **flags**:

```
%%Mexplodemodel <name> [option1] [option2=value] ...
```

The rest of the cell is **Markdown text** where lines starting with `>` are treated as ModelFlow DSL equations. Everything else is documentation that gets rendered in the notebook.

After execution the `Mexplode` instance is available as a variable with the given `<name>`.

### Minimal example

The cell below creates a `Mexplode` object called `simple` and renders the Markdown text in the notebook.

In [ ]:
%%Mexplodemodel simple
A simple two-equation model
> y = c + i
> c = 0.8 * y(-1)

The variable `simple` is now a `Mexplode` instance in the notebook namespace. We can inspect it:

In [ ]:
# Show the expanded equations
simple.show

In [ ]:
# Draw the causal graph
simple.draw

## 2. Available options

Options are placed on the first line after the model name. They follow the pattern `key` (boolean flag) or `key=value`.

| Option | Default | Effect |
|---|---|---|
| `show` | off | Print the expanded equations after creation |
| `draw` | off | Display the causal dependency graph |
| `display` | off | Verbose mode: render Markdown **and** print equations + segment info |
| `render` | on | Render the Markdown text in the cell output |
| `render_list` | on | Include list definitions when rendering (set to `0` to hide them) |
| `latex` | off | Generate a LaTeX PDF of the model specification |
| `spec` | `markdown` | Rendering format (`markdown` by default) |
| `segment` | off | Segmented model building (see Section 4) |
| `replacements` | none | Dimension placeholders, e.g. `replacements=('__d','__{banks}')` |
| `funks` | none | User-defined functions to pass to `Mexplode` |

Boolean options can be turned off explicitly with `option=0` or `option=False`.

### Example: `show` and `draw`

In [ ]:
%%Mexplodemodel demo show draw
A model with tags
><exo,add> y = c + i
> c = 0.8 * y(-1)

### Example: suppressing rendering with `render=0`

In [ ]:
%%Mexplodemodel quiet render=0 show
This text will NOT appear in the output
> a = b + c

## 3. Using lists and `doable` inside the magic

The cell content is passed directly to `Mexplode`, so all DSL features — lists, `do...enddo`, `doable`, `sum()`, tags — work exactly as they would in a normal `Mexplode()` call.

In [ ]:
%%Mexplodemodel bank_model show
## Credit risk model

Define the banks and portfolios:

>list banks  = banks : Danske Nordea /
>list ports  = ports : Mortgage Corporate /

Calculate expected loss per bank and portfolio, then aggregate:

>doable <sum=total> el__{banks}__{ports} = exposure__{banks}__{ports} * pd__{banks}__{ports}

## 4. Segmented models

Large models can be split across **multiple cells** using the `segment=<name>` option. This is one of the most powerful features of the magic.

**How it works:**

1. Each cell with `segment=<name>` stores its content in a dictionary called `<modelname>_dict`.
2. Segments whose name starts with `list` or `text` are treated as definitions/documentation — they are stored but only rendered, not yet compiled.
3. When a non-list/non-text segment is encountered, the magic combines **all list segments** with the **current cell** and builds the `Mexplode` model.
4. A final call **without** `segment` combines **all** stored segments into the complete model.

This lets you build a model incrementally, cell by cell, with explanatory text between sections.

In [ ]:
%%Mexplodemodel stress segment=lists
## List definitions
>list banks  = banks : Alpha Beta /
>list ports  = ports : Retail SME /

In [ ]:
%%Mexplodemodel stress segment=pd_block show
## PD dynamics
Probability of default follows an AR(1) process driven by GDP growth:
>doable <> pd__{banks}__{ports} = 0.02 + 0.8 * pd__{banks}__{ports}(-1) + 0.1 * gdp_growth

In [ ]:
%%Mexplodemodel stress segment=loss_block show
## Expected loss
>doable <sum=total> el__{banks}__{ports} = exposure__{banks}__{ports} * pd__{banks}__{ports}

Now combine all segments into the full model by calling `%%Mexplodemodel` (or `%Mexplodemodel` as a line magic) **without** the `segment` option:

In [ ]:
%%Mexplodemodel stress show draw
## Full stress test model
Combining all segments into one model.

## 5. Using `replacements=` for dimension placeholders

When variable names have many dimensions, you can define a replacement tuple in the notebook and pass it by name:

In [ ]:
repl = ('__d', '__{banks}__{ports}')

In [ ]:
%%Mexplodemodel compact show replacements=repl
>list banks = banks : A B /
>list ports = ports : X Y /
Compact notation using `__d` as placeholder:
>doable <> loss__d = exp__d * pd__d * lgd__d

## 6. The `%%mdmodel` alias

`%%mdmodel` is an exact alias for `%%Mexplodemodel` — same implementation, shorter to type. Both also have **line magic** variants (`%mdmodel`, `%Mexplodemodel`) that rebuild an existing model without adding new cell content.

In [ ]:
%%mdmodel tiny show
> x = y + z

## 7. Tags inside the magic

All `Mexplode` tags work inside the magic cells. Here is a quick reference:

- `<exo>` — allow fixing the variable to an exogenous value
- `<add>` — generate add-factor adjustment variables
- `<fit>` — compute the original fitted value alongside the adjusted one
- `<sum=label>` — inside `doable`, also generate an aggregation equation
- `<add_suffix=__J>` — change the default add-factor suffix from `_A`

Tags are placed between `><` and `>` at the start of a formula line:

In [ ]:
%%Mexplodemodel tagged show
A fully decorated equation:
><exo,add,fit> consumption = 0.8 * income

## 8. Generating a LaTeX PDF

With the `latex` option, the magic converts the cell's Markdown text (headings, tables, bullet lists, equation lines) into LaTeX and compiles a PDF. This requires a LaTeX installation on the system.

```python
%%Mexplodemodel mymodel latex
# My Model Documentation
> y = c + i
```

## 9. How it works internally

The magic is implemented in `modeljupytermagic.py`. Here is the flow:

1. **Parse the first line** — `get_options(line)` splits it into the model name and a dictionary of options.
2. **Handle segments** — if `segment=` is present, the cell text is stored in `<name>_dict[segment]`. List/text segments are rendered and returned early. Other segments combine list definitions with the current cell.
3. **Build the model** — the combined text is passed to `Mexplode(model_text, replacements=..., funks=...)`. This does all DSL expansion: lists, loops, doable, sums, tags.
4. **Push to namespace** — the resulting `Mexplode` instance is injected into the notebook's user namespace via `ip.push()`.
5. **Render output** — depending on options, the magic renders Markdown, prints expanded equations (`.show`), draws the dependency graph (`.draw`), or generates LaTeX.

The key insight is that the magic is a **thin wrapper** — all the heavy lifting is done by `Mexplode`. The magic simply provides a convenient notebook-native interface for defining and documenting models in one place.

## 10. Quick reference

```text
%%Mexplodemodel <name> [show] [draw] [display] [render=0] [latex] [segment=<seg>] [replacements=<var>] [funks=<var>]
<Markdown text with > prefixed DSL equations>
```

After execution: the variable `<name>` holds the `Mexplode` instance with all the usual properties — `.show`, `.draw`, `.mmodel`, `.render`, etc.